In [5]:
!pip install mlflow dagshub dvc dvc-s3 pyarrow fastparquet -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 7.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.5/89.5 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 53.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.4/203.4 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 9.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.4.0 which is incompatible.
datasets 4.0.0 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2026.4.0 which is incompatible.


In [59]:
import pandas as pd
import numpy as np
import dagshub
import mlflow
from getpass import getpass
import os

from sklearn.preprocessing import MinMaxScaler, LabelEncoder, StandardScaler
import torch

In [7]:
!git clone https://github.com/MuhammadShaafImran/Real-Time-Market-Movement-Prediction-System.git
%cd Real-Time-Market-Movement-Prediction-System

Cloning into 'Real-Time-Market-Movement-Prediction-System'...
remote: Enumerating objects: 69, done.
remote: Counting objects: 100% (69/69), done.
remote: Compressing objects: 100% (45/45), done.
remote: Total 69 (delta 21), reused 64 (delta 19), pack-reused 0 (from 0)
Receiving objects: 100% (69/69), 207.83 KiB | 1.91 MiB/s, done.
Resolving deltas: 100% (21/21), done.
/content/Real-Time-Market-Movement-Prediction-System


In [8]:
!ls

LICENSE  README.md


In [10]:
!git checkout dev
!ls

Already on 'dev'
Your branch is up to date with 'origin/dev'.
LICENSE  README.md  requirements.txt  src


In [13]:
# https://dagshub.com/shaafimran257/Real-Time-Market-Movement-Prediction-System.dvc
dagshub.init(repo_owner='shaafimran257', repo_name='Real-Time-Market-Movement-Prediction-System', mlflow=True)
mlflow.set_experiment("Market_Movement_Analysis")
print("MLflow tracking URI:", mlflow.get_tracking_uri())

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=1dc31d6e-d236-45c2-b4bf-535ff2713873&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=01f3dd63c7c1c9df41ef454e33956095108edbfd17960d8cfdc1865a4cd3b1ae




Accessing as RudhanLodhi

Initialized MLflow to track repo "shaafimran257/Real-Time-Market-Movement-Prediction-System"

Repository shaafimran257/Real-Time-Market-Movement-Prediction-System initialized!

2026/05/12 13:25:00 INFO mlflow.tracking.fluent: Experiment with name 'Market_Movement_Analysis' does not exist. Creating a new experiment.


MLflow tracking URI: https://dagshub.com/shaafimran257/Real-Time-Market-Movement-Prediction-System.mlflow


In [24]:
DAGSHUB_USER = "RudhanLodhi"
print("Enter  DagsHub key:")
DAGSHUB_TOKEN = getpass().strip()

os.system('dvc remote modify origin --local auth basic')
os.system(f'dvc remote modify origin --local user {DAGSHUB_USER}')
os.system(f'dvc remote modify origin --local password {DAGSHUB_TOKEN}')

exit_code = os.system('dvc pull')

if exit_code == 0:
    print("✅ Success! Your images have been downloaded.")
else:
    print("❌ Error: dvc pull failed. ")

Enter  DagsHub key:
✅ Success! Your images have been downloaded.


In [25]:
!ls src/data

processed  processed.dvc  raw  raw.dvc


In [29]:
dataset_path = 'src/data/processed/latest_ml_dataset_v4_finbert.parquet'
df = pd.read_parquet(dataset_path)
df.head()

,timestamp,symbol,open,high,low,close,volume,RSI,MACD,MACD_signal,SMA_20,EMA_20,BB_high,BB_low,ticker_sentiment,news_count,market_sentiment,reddit_hype,label
0,2026-02-13 14:30:00+00:00,AAPL,262.010010,262.230011,258.799988,259.299988,4578357,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,1
1,2026-02-13 14:35:00+00:00,AAPL,259.339996,260.260010,258.799988,259.385010,772714,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,1
2,2026-02-13 14:40:00+00:00,AAPL,259.350006,260.190002,259.019989,259.519989,646951,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,1
3,2026-02-13 14:45:00+00:00,AAPL,259.510010,260.980011,259.399994,260.274994,611692,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0
4,2026-02-13 14:50:00+00:00,AAPL,260.269989,260.326691,258.820007,259.500000,564613,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0


In [30]:
print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")

Dataset shape: (14037, 19)

Columns: ['timestamp', 'symbol', 'open', 'high', 'low', 'close', 'volume', 'RSI', 'MACD', 'MACD_signal', 'SMA_20', 'EMA_20', 'BB_high', 'BB_low', 'ticker_sentiment', 'news_count', 'market_sentiment', 'reddit_hype', 'label']


In [31]:
print(f"\nData types:")
print(df.dtypes)


Data types:
timestamp           datetime64[ns, UTC]
symbol                           object
open                            float64
high                            float64
low                             float64
close                           float64
volume                            int64
RSI                             float64
MACD                            float64
MACD_signal                     float64
SMA_20                          float64
EMA_20                          float64
BB_high                         float64
BB_low                          float64
ticker_sentiment                float64
news_count                      float64
market_sentiment                float64
reddit_hype                     float64
label                             int32
dtype: object


In [32]:
print(f"\nMissing values:")
print(df.isnull().sum().to_frame("missing").query("missing > 0"))
print(f"\nTarget distribution:")
print(df['label'].value_counts())


Missing values:
             missing
RSI               39
MACD              75
MACD_signal       99
SMA_20            57
EMA_20            57
BB_high           57
BB_low            57

Target distribution:
label
0    7019
1    7018
Name: count, dtype: int64


## Preprocessing

In [40]:
df_sorted = df.sort_values('timestamp').reset_index(drop=True)

y = df_sorted['label'].values
X = df_sorted.drop(['label', 'timestamp'], axis=1)

categorical_cols = X.select_dtypes(include=['object', 'string', 'category']).columns.tolist()
numeric_cols = X.select_dtypes(include=['number']).columns.tolist()

print(f"Categorical columns: {categorical_cols}")
print(f"Numeric columns: {numeric_cols}")

Categorical columns: ['symbol']
Numeric columns: ['open', 'high', 'low', 'close', 'volume', 'RSI', 'MACD', 'MACD_signal', 'SMA_20', 'EMA_20', 'BB_high', 'BB_low', 'ticker_sentiment', 'news_count', 'market_sentiment', 'reddit_hype']


In [54]:
X_processed = X.copy()

for col in numeric_cols:
    if X_processed[col].isnull().sum() > 0:
        X_processed[col] = X_processed[col].ffill().bfill().fillna(X_processed[col].mean())

for col in categorical_cols:
    X_processed[col] = X_processed[col].fillna("Unknown").astype(str)
    X_processed[col] = LabelEncoder().fit_transform(X_processed[col])

y_raw = y
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_raw)

print(f"\nProcessed features shape: {X_processed.shape}")
print(f"Missing values after preprocessing: {X_processed.isnull().sum().sum()}")
print(f"Target shape: {y.shape}")
print(f"\nFeature statistics:")
print(X_processed.describe())


Processed features shape: (14037, 17)
Missing values after preprocessing: 0
Target shape: (14037,)

Feature statistics:
             symbol          open          high           low         close  \
count  14037.000000  14037.000000  14037.000000  14037.000000  14037.000000   
mean       1.000000    324.562520    324.987535    324.137906    324.572444   
std        0.816526     55.719987     55.871132     55.566657     55.727084   
min        0.000000    245.509995    245.949997    245.509995    245.529999   
25%        0.000000    270.609985    270.915009    270.399994    270.619995   
50%        1.000000    308.541687    308.859985    308.172699    308.540009   
75%        2.000000    382.489990    383.100006    381.850006    382.529999   
max        2.000000    447.989990    449.160004    447.519989    447.980011   

             volume           RSI          MACD   MACD_signal        SMA_20  \
count  1.403700e+04  14037.000000  14037.000000  14037.000000  14037.000000   
mean   5.

In [55]:
n_samples = len(X_processed)
train_size = int(0.7 * n_samples)  # 70% for training
val_size = int(0.1 * n_samples)    # 10% for validation
test_size = n_samples - train_size - val_size  # 20% for testing

# Split chronologically
X_train = X_processed.iloc[:train_size].values
y_train = y[:train_size]

X_val = X_processed.iloc[train_size:train_size + val_size].values
y_val = y[train_size:train_size + val_size]

X_test = X_processed.iloc[train_size + val_size:].values
y_test = y[train_size + val_size:]

print(f"Train set: {X_train.shape}, {y_train.shape}")
print(f"Validation set: {X_val.shape}, {y_val.shape}")
print(f"Test set: {X_test.shape}, {y_test.shape}")
print(f"\nClass distribution in train: {np.bincount(y_train)}")
print(f"Class distribution in val: {np.bincount(y_val)}")
print(f"Class distribution in test: {np.bincount(y_test)}")


Train set: (9825, 17), (9825,)
Validation set: (1403, 17), (1403,)
Test set: (2809, 17), (2809,)

Class distribution in train: [4919 4906]
Class distribution in val: [719 684]
Class distribution in test: [1381 1428]


In [56]:
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)
feature_names = X_processed.columns.tolist()

print(f"\nFeatures scaled successfully")
print(f"Train mean: {X_train_scaled.mean(axis=0)[:5]}, std: {X_train_scaled.std(axis=0)[:5]}")
print(f"Validation shape: {X_val_scaled.shape}, Test shape: {X_test_scaled.shape}")


Features scaled successfully
Train mean: [0.5        0.39758287 0.39742128 0.39687142 0.39754442], std: [0.40824829 0.30896935 0.30975285 0.30940698 0.30901628]
Validation shape: (1403, 17), Test shape: (2809, 17)


In [57]:
def create_sequences(X, y, seq_length=30):
    """
    Create sequences for time-series models.
    
    Args:
        X: Feature matrix (n_samples, n_features)
        y: Target labels (n_samples,)
        seq_length: Length of each sequence
    
    Returns:
        X_seq: Sequences (n_sequences, seq_length, n_features)
        y_seq: Corresponding labels (n_sequences,)
    """
    X_seq, y_seq = [], []
    for i in range(len(X) - seq_length):
        X_seq.append(X[i:i + seq_length])
        # Use the label of the next time step as target
        y_seq.append(y[i + seq_length])
    return np.array(X_seq), np.array(y_seq)

SEQ_LENGTH = 30  # Use 30 previous time steps to predict the next

# Generate sequences
X_train_seq, y_train_seq = create_sequences(X_train_scaled, y_train, SEQ_LENGTH)
X_val_seq, y_val_seq = create_sequences(X_val_scaled, y_val, SEQ_LENGTH)
X_test_seq, y_test_seq = create_sequences(X_test_scaled, y_test, SEQ_LENGTH)

print(f"Sequence generation completed with sequence length: {SEQ_LENGTH}")
print(f"Train sequences: {X_train_seq.shape}, labels: {y_train_seq.shape}")
print(f"Val sequences: {X_val_seq.shape}, labels: {y_val_seq.shape}")
print(f"Test sequences: {X_test_seq.shape}, labels: {y_test_seq.shape}")
print(f"Input features per timestep: {X_train_seq.shape[2]}")

Sequence generation completed with sequence length: 30
Train sequences: (9795, 30, 17), labels: (9795,)
Val sequences: (1373, 30, 17), labels: (1373,)
Test sequences: (2779, 30, 17), labels: (2779,)
Input features per timestep: 17


In [60]:
X_train_tensor = torch.FloatTensor(X_train_seq)
y_train_tensor = torch.LongTensor(y_train_seq)

X_val_tensor = torch.FloatTensor(X_val_seq)
y_val_tensor = torch.LongTensor(y_val_seq)

X_test_tensor = torch.FloatTensor(X_test_seq)
y_test_tensor = torch.LongTensor(y_test_seq)

print(f"\nTensors created successfully")
print(f"Device availability - CUDA: {torch.cuda.is_available()}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


Tensors created successfully
Device availability - CUDA: False
Using device: cpu
